# Build Dataset

This notebook converts the raw market data collected from the Binance API into a structured dataset suitable for analysis and modelling.

The raw data contains order book snapshots and trade data stored in parquet files. In this notebook we:

- load the raw data
- unpack order book levels into columns
- compute basic market variables such as mid price and spread
- aggregate trades around each snapshot

The result is two clean datasets that will be used for feature engineering and modelling in later notebooks and then combined.

# Loading in the raw dataset

In [16]:
import glob
import pandas as pd
import numpy as np
import math
import black
from microstructure_alpha.utils.logger import setup_logger

pd.set_option("display.max_columns", None)

In [17]:
logger = setup_logger(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\logs\\building_dataset_notebook.log"
)

In [18]:
files_trades = glob.glob(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\raw\\tester_folder\\trade_data_raw_*"
)
files_lob = glob.glob(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\raw\\tester_folder\\lob_data_raw_*"
)

lob_df = pd.concat([pd.read_parquet(f) for f in files_lob])
trade_df = pd.concat([pd.read_parquet(f) for f in files_trades])

# Create Lob dataset

Helper function to explode limit order bokok and seperate volume and price

In [19]:
def explode_lob(df, side: str, level: int = 10):

    df = df.copy()
    df[side] = df[side].str[:level]

    out = df[["timestamp", "lastUpdateId", side]].explode(side)

    out[f"lob_{side}_price"], out[f"lob_{side}_volume"] = zip(*out[side])
    out[f"lob_{side}_price"] = out[f"lob_{side}_price"].astype(float)
    out[f"lob_{side}_volume"] = out[f"lob_{side}_volume"].astype(float)

    out["level"] = out.groupby("timestamp").cumcount() + 1
    out = out.drop(columns=side)
    return out

In [20]:
bids_df_10 = explode_lob(lob_df, "bids")
asks_df_10 = explode_lob(lob_df, "asks")
bids_df_10

,timestamp,lastUpdateId,lob_bids_price,lob_bids_volume,level
0,1773124136240,89805309318,70158.63,2.42879,1
0,1773124136240,89805309318,70158.62,0.00048,2
0,1773124136240,89805309318,70158.61,0.12319,3
0,1773124136240,89805309318,70158.60,0.05000,4
0,1773124136240,89805309318,70158.56,0.00016,5
...,...,...,...,...,...
499,1773124134593,89805308027,70152.80,0.05016,6
499,1773124134593,89805308027,70152.79,0.35230,7
499,1773124134593,89805308027,70152.78,0.00008,8
499,1773124134593,89805308027,70151.83,0.00016,9


## Create a new column for each level of price and volume

In [21]:
def pivot_helper(df, side: str):
    prices = df.pivot(index="timestamp", columns="level", values=f"lob_{side}_price")
    volume = df.pivot(index="timestamp", columns="level", values=f"lob_{side}_volume")

    prices.columns = [f"lob_{side}_price_{i}" for i in prices.columns]
    volume.columns = [f"lob_{side}_volume_{i}" for i in volume.columns]

    prices.reset_index()
    volume.reset_index()

    return pd.concat([prices, volume], axis=1)

In [22]:
combined_lob_df = pd.concat(
    [
        pivot_helper(bids_df_10, "bids"),
        pivot_helper(asks_df_10, "asks"),
    ],
    axis=1,
)
combined_lob_df.reset_index(inplace=True)

# Create Trades Dataset

Correct dtypes of trades dataset

In [23]:
trade_df = trade_df.astype(
    {
        "price": float,
        "qty": float,
        "quoteQty": float,
        "time": "int64",
        "isBuyerMaker": bool,
        "isBestMatch": bool,
    }
)

# pair up trades with the correct lob snapshot

In [24]:
trade_df = trade_df.sort_values("time")
combined_lob_df = combined_lob_df.sort_values("timestamp")

trade_df = pd.merge_asof(
    trade_df,
    combined_lob_df[["timestamp"]],
    left_on="time",
    right_on="timestamp",
    direction="backward",
    tolerance=1000
)

trade_df = trade_df.dropna(subset=["timestamp"])

assert (trade_df["time"] < trade_df["timestamp"]).sum() == 0

# Save both dataframes to interim location

In [25]:
lob_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\interim\\lob_snapshot_df.parquet"
trade_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\interim\\trades.parquet"

combined_lob_df.to_parquet(lob_path, compression="snappy")
trade_df.to_parquet(trade_path, compression="snappy")

# Batch Version for bigger datasets

## Define the helper functions

In [26]:
def explode_lob(df, side: str, level: int = 10):

    df = df.copy()
    df[side] = df[side].str[:level]

    out = df[["timestamp", "lastUpdateId", side]].explode(side)

    out[f"lob_{side}_price"], out[f"lob_{side}_volume"] = zip(*out[side])
    out[f"lob_{side}_price"] = out[f"lob_{side}_price"].astype(float)
    out[f"lob_{side}_volume"] = out[f"lob_{side}_volume"].astype(float)

    out["level"] = out.groupby("timestamp").cumcount() + 1
    out = out.drop(columns=side)
    return out


def pivot_helper(df, side: str):
    prices = df.pivot(index="timestamp", columns="level", values=f"lob_{side}_price")
    volume = df.pivot(index="timestamp", columns="level", values=f"lob_{side}_volume")

    prices.columns = [f"lob_{side}_price_{i}" for i in prices.columns]
    volume.columns = [f"lob_{side}_volume_{i}" for i in volume.columns]

    prices.reset_index()
    volume.reset_index()

    return pd.concat([prices, volume], axis=1)

## Define the main functions to loop over to process in batches

In [27]:
def process_lob(lob_df):
    bids_df_10 = explode_lob(lob_df, "bids")
    asks_df_10 = explode_lob(lob_df, "asks")
    combined_lob_df = pd.concat(
        [
            pivot_helper(bids_df_10, "bids"),
            pivot_helper(asks_df_10, "asks"),
        ],
        axis=1,
    )
    combined_lob_df.reset_index(inplace=True)
    return combined_lob_df


def process_trade(trade_df, combined_lob_df):

    trade_df = trade_df.astype(
        {
            "price": float,
            "qty": float,
            "quoteQty": float,
            "time": "int64",
            "isBuyerMaker": bool,
            "isBestMatch": bool,
        }
    )

    trade_df = trade_df.sort_values("time")
    combined_lob_df = combined_lob_df.sort_values("timestamp")

    trade_df = pd.merge_asof(
        trade_df,
        combined_lob_df[["timestamp"]],
        left_on="time",
        right_on="timestamp",
        direction="backward",
        tolerance=1000
    )

    trade_df = trade_df.dropna(subset=["timestamp"])

    assert (trade_df["time"] < trade_df["timestamp"]).sum() == 0

    return trade_df

## Create batch pipeline

In [28]:
import glob
from pathlib import Path
import gc
from tqdm import tqdm

In [29]:
def run_batch_build_dataset_pipeline(lob_path: str, trade_path: str, save_path: str):
    files_trade = sorted(glob.glob(trade_path + "trades_data_raw_*"))
    files_lob = sorted(glob.glob(lob_path + "lob_data_raw_*"))

    logger.info("Starting batch pipeline")
    logger.info(f"Found {len(files_trade)} trade files")
    logger.info(f"Found {len(files_lob)} lob files")

    total_batches = min(len(files_trade), len(files_lob))
    logger.info(f"Processing {total_batches} batches")

    for i, (trade_file, lob_file) in tqdm(
        enumerate(zip(files_trade, files_lob)),
        total=total_batches,
        desc="Processing batches",
    ):
        lob_df = pd.read_parquet(lob_file)
        trade_df = pd.read_parquet(trade_file)
        processed_lob = process_lob(lob_df)
        processed_trade = process_trade(trade_df, processed_lob)

        processed_lob.to_parquet(save_path + f"lob_interim_{i}.parquet", compression="snappy")
        processed_trade.to_parquet(
            save_path + f"trade_interim_{i}.parquet", compression="snappy"
        )

        del lob_df
        del trade_df
        del processed_lob
        del processed_trade

        gc.collect()
        #if i > 3:
        #    return
    logger.info("Batch pipeline completed successfully")
    return

In [30]:
trade_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\raw\\trades\\"
lob_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\raw\\lob\\"

save_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\interim\\"

RUN_PIPELINE = True
if RUN_PIPELINE:
    run_batch_build_dataset_pipeline(lob_path, trade_path, save_path)

2026-03-29 19:05:17 | INFO | Starting batch pipeline
2026-03-29 19:05:17 | INFO | Found 89 trade files
2026-03-29 19:05:17 | INFO | Found 89 lob files
2026-03-29 19:05:17 | INFO | Processing 89 batches


Processing batches: 100%|██████████| 89/89 [00:07<00:00, 11.20it/s]

2026-03-29 19:05:25 | INFO | Batch pipeline completed successfully
